# 4. Results Analysis

**Purpose**: Understand results, identify issues, prepare final outputs.

## Sections
1. Load predictions
2. Match rate by source
3. Confidence distribution
4. Sample matches at each tier
5. Unmatched records analysis
6. Export final outputs


---
## 1. Setup and Load Data


In [8]:
import sys
sys.path.insert(0, '/Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test')

import pandas as pd
import numpy as np
from pathlib import Path

from config import config
from utils import (
    log_step, 
    load_checkpoint,
    save_json
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)

print("Imports loaded successfully")


Imports loaded successfully


In [9]:
# Load predictions
predictions_df = load_checkpoint(config.paths.PREDICTIONS, "Predictions")

if predictions_df is None:
    raise FileNotFoundError(f"Predictions not found at {config.paths.PREDICTIONS}. Run 3_inference.ipynb first.")

print(f"\nPREDICTIONS LOADED")
print("=" * 50)
print(f"Total predictions: {len(predictions_df):,}")
print(f"Columns: {list(predictions_df.columns)}")


[17:10:19] [!] Checkpoint not found: /Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test/data/predictions.parquet


FileNotFoundError: Predictions not found at /Users/robertlalani/Desktop/entity_resolution_12_18_25/org_claude_test/data/predictions.parquet. Run 3_inference.ipynb first.

In [3]:
# Load reference data for name lookups
dim_org_df = load_checkpoint(config.paths.DATA_DIR + "/dim_org_training.parquet", "dim_org")

if dim_org_df is not None:
    # Create lookup dictionaries
    dim_org_names = dict(zip(dim_org_df['unique_id'], dim_org_df['name']))
    print(f"Loaded dim_org lookup: {len(dim_org_names):,} records")
else:
    dim_org_names = {}
    print("Warning: Could not load dim_org for name lookups")


[13:54:31]  Loading checkpoint: dim_org
[13:54:31]    Loaded 109,851 rows
Loaded dim_org lookup: 109,851 records


---
## 2. Cluster Analysis


In [4]:
# Load clusters
clusters_df = load_checkpoint(config.paths.CLUSTERS, "Entity clusters")

if clusters_df is not None:
    print("CLUSTER SIZE DISTRIBUTION")
    print("=" * 50)
    
    cluster_sizes = clusters_df.groupby('cluster_id').size()
    
    # Basic stats
    print(f"\nTotal clusters: {len(cluster_sizes):,}")
    print(f"Total records in clusters: {len(clusters_df):,}")
    
    # Size categories
    singleton = (cluster_sizes == 1).sum()
    size_2 = (cluster_sizes == 2).sum()
    size_3_5 = ((cluster_sizes >= 3) & (cluster_sizes <= 5)).sum()
    size_6_10 = ((cluster_sizes >= 6) & (cluster_sizes <= 10)).sum()
    size_large = (cluster_sizes > 10).sum()
    
    print(f"\nCluster Size Distribution:")
    print(f"  Singletons (1 record):    {singleton:>8,} ({100*singleton/len(cluster_sizes):.1f}%)")
    print(f"  Pairs (2 records):        {size_2:>8,} ({100*size_2/len(cluster_sizes):.1f}%)")
    print(f"  Small (3-5 records):      {size_3_5:>8,} ({100*size_3_5/len(cluster_sizes):.1f}%)")
    print(f"  Medium (6-10 records):    {size_6_10:>8,} ({100*size_6_10/len(cluster_sizes):.1f}%)")
    print(f"  Large (>10 records):      {size_large:>8,} ({100*size_large/len(cluster_sizes):.1f}%)")
    
    print(f"\nLargest cluster: {cluster_sizes.max()} records")
    print(f"Average cluster size: {cluster_sizes.mean():.2f}")
    print(f"Median cluster size: {cluster_sizes.median():.1f}")
else:
    print("No clusters found. Run 3_inference.ipynb with clustering enabled.")


[13:54:33]  Loading checkpoint: Entity clusters
[13:54:33]    Loaded 2,632 rows
CLUSTER SIZE DISTRIBUTION

Total clusters: 672
Total records in clusters: 2,632

Cluster Size Distribution:
  Singletons (1 record):           0 (0.0%)
  Pairs (2 records):             436 (64.9%)
  Small (3-5 records):           190 (28.3%)
  Medium (6-10 records):          22 (3.3%)
  Large (>10 records):            24 (3.6%)

Largest cluster: 303 records
Average cluster size: 3.92
Median cluster size: 2.0


In [5]:
# Inspect largest clusters (may indicate problematic matches)
if clusters_df is not None and len(cluster_sizes) > 0:
    print("LARGEST CLUSTERS")
    print("=" * 50)
    print("(Large clusters may indicate over-matching)")
    
    largest_cluster_ids = cluster_sizes.nlargest(5).index
    
    for cluster_id in largest_cluster_ids:
        cluster_records = clusters_df[clusters_df['cluster_id'] == cluster_id]
        size = len(cluster_records)
        
        print(f"\nCluster {cluster_id} ({size} records):")
        
        # Show names if available
        if 'name' in cluster_records.columns:
            names = cluster_records['name'].head(5).tolist()
        elif 'unique_id' in cluster_records.columns and dim_org_names:
            names = [dim_org_names.get(uid, uid) for uid in cluster_records['unique_id'].head(5)]
        else:
            names = cluster_records['unique_id'].head(5).tolist()
        
        for name in names:
            print(f"  - {str(name)[:70]}")
        if size > 5:
            print(f"  ... and {size - 5} more")


LARGEST CLUSTERS
(Large clusters may indicate over-matching)

Cluster 1 (303 records):
  - mis_c96d540b401324ea231528d89f23d845
  - mis_8e107f8dee181f9d298b5eb1c220c8c5
  - mis_e7abb52cc4e6da3b189fbbf6f4a978cb
  - mis_53dbc23cd205cd14e17dc4dfcadff1a9
  - mis_fe522d354f565739ebcd6335fd3bd486
  ... and 298 more

Cluster 15 (180 records):
  - mis_864ed0f2cdf212ecb34d1d9479e7a580
  - mis_fee820cbc2b8d8ca75d154b7ebba03ea
  - mis_b2fdc7c185f7734a761dfa4d318a4e19
  - mis_d4042eb876645a7baa77ef5f15e597ef
  - mis_0864751d1ed98817d8f65db65b2b4306
  ... and 175 more

Cluster 23 (81 records):
  - mis_ba8068b9d040aa060cb35536bf539760
  - mis_5b5d1241697869eca2866e2480139411
  - mis_360048dfdacda051ddec2747e9057dd3
  - mis_355f894e07132fea2ab8fd65128fadee
  - mis_46c4de5af8f9d16475f923b0793fa8e3
  ... and 76 more

Cluster 8 (60 records):
  - mis_f7ddc05612dc6b73365de19c6e8f9c4b
  - mis_f61a00c65ba83dd6bfdda5f186957745
  - mis_6ef123c1440f6332a3039841c4fe446d
  - mis_0ad64438459712705076144e76fae779


---
## 3. Splink Model Visualizations


In [6]:
# Load model for visualizations
import json
from splink import Linker, DuckDBAPI
from utils import load_json

model_json = load_json(config.paths.MODEL_FILE, "Trained model")

if model_json is not None:
    # Remove training metadata (not a Splink setting)
    model_settings = {k: v for k, v in model_json.items() if k != 'training_metadata'}
    
    # Initialize linker for visualizations
    linker = Linker([dim_org_df], model_settings, db_api=DuckDBAPI())
    print("Linker initialized for visualizations")
else:
    linker = None
    print("Model not found - visualizations unavailable")


SETTINGS VALIDATION: Errors were identified in your settings dictionary. 

Invalid Columns(s) in Blocking Rule(s)

    SQL: `l.distinctive_token = r.distinctive_token AND l.city = r.city`
       - Missing column(s) from input dataframe(s): `distinctive_token`

Invalid Columns(s) in Comparison(s)

Comparison: phonetic_match
--------------------------------------
    SQL: `"distinctive_soundex_l" IS NULL OR "distinctive_soundex_r" IS NULL`
       - Missing column(s) from input dataframe(s): `distinctive_soundex`

    SQL: `"distinctive_soundex_l" = "distinctive_soundex_r"`
       - Missing column(s) from input dataframe(s): `distinctive_soundex`

Comparison: distinctive_match
--------------------------------------
    SQL: `"distinctive_tokens_l" IS NULL OR "distinctive_tokens_r" IS NULL`
       - Missing column(s) from input dataframe(s): `distinctive_tokens`

    SQL: `array_length(distinctive_tokens_l, 1) = 0 OR array_length(distinctive_tokens_r, 1) = 0`
       - Missing column(s) fro

[13:54:53]  Loading JSON: Trained model
Linker initialized for visualizations


In [7]:
# Match weights chart - shows contribution of each comparison
if linker is not None:
    print("MATCH WEIGHTS CHART")
    print("=" * 50)
    print("Shows how each comparison contributes to match probability")
    print("Positive weights support match, negative weights oppose\n")
    
    linker.visualisations.match_weights_chart()


MATCH WEIGHTS CHART
Shows how each comparison contributes to match probability
Positive weights support match, negative weights oppose



In [8]:
# M/U probability chart - shows parameter estimates
if linker is not None:
    print("M AND U PROBABILITY CHART")
    print("=" * 50)
    print("M = probability of agreement given records match")
    print("U = probability of random agreement (non-match)\n")
    
    linker.visualisations.m_u_parameters_chart()


M AND U PROBABILITY CHART
M = probability of agreement given records match
U = probability of random agreement (non-match)



In [9]:
# Waterfall chart for a sample prediction
# Shows how each comparison contributes to the final match probability
if linker is not None and len(predictions_df) > 0:
    print("WATERFALL CHART (Sample High-Confidence Match)")
    print("=" * 50)
    
    # Get a sample high-confidence prediction
    sample_pred = predictions_df[predictions_df['match_probability'] > 0.95].head(1)
    
    if len(sample_pred) > 0:
        print(f"Sample pair (prob={sample_pred['match_probability'].values[0]:.4f}):\n")
        
        # Convert to format linker expects
        linker.visualisations.waterfall_chart(sample_pred.to_dict('records'))


WATERFALL CHART (Sample High-Confidence Match)
Sample pair (prob=1.0000):



ValueError: retain_intermediate_calculation_columns and retain_matching_columns must both be set to True in your settings dictionary to use this function, because otherwise the necessary columns will not be available in the input records. Their current values are False and True, respectively. Please re-run your linkage with them both set to True.

---
## 2. Match Rate by Source


In [10]:
# Match rate by source table
print("MATCH RATE BY SOURCE TABLE")
print("=" * 70)

if 'source_table' in predictions_df.columns:
    source_stats = predictions_df.groupby('source_table').agg({
        'match_probability': ['count', 'mean', 'median', 'max']
    }).round(3)
    source_stats.columns = ['count', 'mean_prob', 'median_prob', 'max_prob']
    source_stats = source_stats.sort_values('count', ascending=False)
    
    print(source_stats.to_string())
else:
    print("No source_table column in predictions")


MATCH RATE BY SOURCE TABLE
                                                       count  mean_prob  median_prob  max_prob
source_table                                                                                  
nih_clinical_trials_gov_silver.cl_trial_organizations    127      1.000          1.0       1.0
nih_clinical_trials_gov_silver.cl_trial_locations        105      0.988          1.0       1.0
nih_clinical_trials_gov_silver.cl_trial_collaborators     78      1.000          1.0       1.0
who_clinical_trials_silver.studies_metadata               57      0.989          1.0       1.0
open_fda_silver.ndc_drugs                                 46      1.000          1.0       1.0
uspto_silver.patents                                      25      1.000          1.0       1.0
chinese_clinical_trials_silver.trial_contacts             14      1.000          1.0       1.0
chinese_clinical_trials_silver.trials                     14      1.000          1.0       1.0
nih_clinical_trials_gov

---
## 3. Confidence Distribution


In [11]:
# Confidence tier distribution
print("CONFIDENCE TIER DISTRIBUTION")
print("=" * 50)

# Define tiers
predictions_df['confidence_tier'] = pd.cut(
    predictions_df['match_probability'],
    bins=[0, 0.5, 0.7, 0.85, 0.95, 1.0],
    labels=['<0.5 (Very Low)', '0.5-0.7 (Low)', '0.7-0.85 (Medium)', '0.85-0.95 (High)', '0.95-1.0 (Very High)']
)

tier_counts = predictions_df['confidence_tier'].value_counts().sort_index()

print("\nDistribution:")
for tier, count in tier_counts.items():
    pct = 100 * count / len(predictions_df)
    bar = '#' * int(pct / 2)
    print(f"  {tier:<25} | {bar:<50} {count:>8,} ({pct:5.1f}%)")


CONFIDENCE TIER DISTRIBUTION

Distribution:
  <0.5 (Very Low)           |                                                           0 (  0.0%)
  0.5-0.7 (Low)             |                                                           1 (  0.2%)
  0.7-0.85 (Medium)         |                                                           0 (  0.0%)
  0.85-0.95 (High)          | #                                                        12 (  2.4%)
  0.95-1.0 (Very High)      | ################################################        480 ( 97.4%)


In [12]:
# Basic statistics
print("\nMATCH PROBABILITY STATISTICS")
print("=" * 50)
print(predictions_df['match_probability'].describe())



MATCH PROBABILITY STATISTICS
count    493.000000
mean       0.996160
std        0.021028
min        0.663164
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: match_probability, dtype: float64


---
## 4. Sample Matches at Each Tier


In [13]:
def show_sample_matches(df, tier_name, n=5):
    """Display sample matches from a tier"""
    print(f"\n{tier_name}")
    print("-" * 70)
    
    sample = df.head(n)
    for i, (_, row) in enumerate(sample.iterrows()):
        left_id = row.get('unique_id_l', 'N/A')
        right_id = row.get('unique_id_r', 'N/A')
        prob = row.get('match_probability', 0)
        
        # Get names from lookup
        left_name = dim_org_names.get(left_id, left_id)
        right_name = dim_org_names.get(right_id, right_id)
        
        print(f"\n  Match {i+1} (prob={prob:.4f}):")
        print(f"    L: {str(left_name)[:65]}")
        print(f"    R: {str(right_name)[:65]}")


In [14]:
# Very high confidence matches (>0.95)
very_high = predictions_df[predictions_df['match_probability'] > 0.95].sort_values('match_probability', ascending=False)
show_sample_matches(very_high, "VERY HIGH CONFIDENCE (>0.95)")



VERY HIGH CONFIDENCE (>0.95)
----------------------------------------------------------------------

  Match 1 (prob=1.0000):
    L: Tianjin University
    R: mis_1b0a5aed300e674c8192dd75e53fc0ba

  Match 2 (prob=1.0000):
    L: Shandong Provincial Institute of Dermatology and Venereology
    R: mis_2d395c207ba479bdccfbefd362085ac6

  Match 3 (prob=1.0000):
    L: Longhua Hospital Shanghai University of Traditional Chinese Medic
    R: mis_82b1b171af470825cf4988272c65b770

  Match 4 (prob=1.0000):
    L: Xinjiang Medical University
    R: mis_355d0cf3c1622a0847927e528353935c

  Match 5 (prob=1.0000):
    L: Shanghai University of Traditional Chinese Medicine
    R: mis_47be5994453ada6e6f76b334c7fff9fb


In [15]:
# Medium confidence matches (0.7-0.85) - these may need review
medium = predictions_df[(predictions_df['match_probability'] >= 0.7) & (predictions_df['match_probability'] < 0.85)]
show_sample_matches(medium, "MEDIUM CONFIDENCE (0.7-0.85) - May Need Review")



MEDIUM CONFIDENCE (0.7-0.85) - May Need Review
----------------------------------------------------------------------


In [16]:
# Low confidence matches (0.5-0.7) - likely false positives
low = predictions_df[(predictions_df['match_probability'] >= 0.5) & (predictions_df['match_probability'] < 0.7)]
show_sample_matches(low, "LOW CONFIDENCE (0.5-0.7) - Likely False Positives")



LOW CONFIDENCE (0.5-0.7) - Likely False Positives
----------------------------------------------------------------------

  Match 1 (prob=0.6632):
    L: Billings Clinic Cody
    R: mis_daab05eac94ef69836446e5da3d6a41d


---
## 5. Quality Analysis


---
## 5. Quality Analysis


In [17]:
# Identify potential issues
print("QUALITY ANALYSIS")
print("=" * 50)

# Check for multiple matches to same dim_org record
if 'unique_id_l' in predictions_df.columns:
    match_counts = predictions_df.groupby('unique_id_l').size()
    multiple_matches = match_counts[match_counts > 1]
    
    print(f"\nRecords with multiple matches: {len(multiple_matches):,}")
    if len(multiple_matches) > 0:
        print(f"  Max matches to single record: {multiple_matches.max()}")
        print(f"\nTop records with most matches:")
        for uid, count in multiple_matches.nlargest(5).items():
            name = dim_org_names.get(uid, uid)
            print(f"  {count}x | {str(name)[:60]}")


QUALITY ANALYSIS

Records with multiple matches: 51
  Max matches to single record: 205

Top records with most matches:
  205x | National Cancer Institute
  46x | Boiron (France)
  15x | Apple (Israel)
  12x | Dartmouth–Hitchcock Medical Center
  10x | Xiangya Hospital Central South University


---
## 6. Export Final Outputs


In [18]:
# Create final output with different confidence levels
print("CREATING FINAL OUTPUTS")
print("=" * 50)

# High confidence matches only (production ready)
high_conf = predictions_df[predictions_df['match_probability'] >= config.matching.THRESHOLD_HIGH_CONFIDENCE].copy()

# Add human-readable names
if dim_org_names:
    high_conf['matched_org_name'] = high_conf['unique_id_l'].map(dim_org_names)

# Select final columns
output_cols = ['unique_id_r', 'unique_id_l', 'match_probability', 'source_table']
if 'matched_org_name' in high_conf.columns:
    output_cols.append('matched_org_name')

final_output = high_conf[output_cols].rename(columns={
    'unique_id_r': 'mismatched_id',
    'unique_id_l': 'dim_org_id',
    'match_probability': 'confidence'
})

print(f"High confidence matches: {len(final_output):,}")
print(f"\nSample output:")
display(final_output.head(10))


CREATING FINAL OUTPUTS
High confidence matches: 480

Sample output:


,mismatched_id,dim_org_id,confidence,source_table,matched_org_name
2,mis_1c5bb770c50fbb2b3c7b761b4774f5e6,dim_ASC-OR-0000000078724-1.0-1724880256,1.0,chinese_clinical_trials_silver.trial_contacts,Xiangya Hospital Central South University
3,mis_6214da8436fdf98fc3ca6f5febf429d3,dim_ASC-OR-0000000078724-1.0-1724880256,1.0,chinese_clinical_trials_silver.trial_contacts,Xiangya Hospital Central South University
4,mis_c8d2070f5fd8fab704b1cc30dc841b8d,dim_ASC-OR-0000000078724-1.0-1724880256,1.0,chinese_clinical_trials_silver.trial_contacts,Xiangya Hospital Central South University
5,mis_b231ca0ce7de27c4c9dfd3fc50be6ef6,dim_ASC-OR-0000000078724-1.0-1724880256,1.0,chinese_clinical_trials_silver.trial_contacts,Xiangya Hospital Central South University
6,mis_81bb1185ea470ba37e26f5b62acdef1b,dim_ASC-OR-0000000078724-1.0-1724880256,1.0,chinese_clinical_trials_silver.trial_contacts,Xiangya Hospital Central South University
7,mis_0ec5aa3f3e6320615e22586c4d26af07,dim_ASC-OR-0000000078724-1.0-1724880256,1.0,chinese_clinical_trials_silver.trial_contacts,Xiangya Hospital Central South University
8,mis_6d709fe46063a821dff8275aaae0c67f,dim_ASC-OR-0000000078724-1.0-1724880256,1.0,chinese_clinical_trials_silver.trial_contacts,Xiangya Hospital Central South University
9,mis_37f56c17d441fae5763933d569ea4a84,dim_ASC-OR-0000000078724-1.0-1724880256,1.0,chinese_clinical_trials_silver.trial_contacts,Xiangya Hospital Central South University
10,mis_d4e966a7fe321f818cf5e4421b785ba1,dim_ASC-OR-0000000078724-1.0-1724880256,1.0,chinese_clinical_trials_silver.trials,Xiangya Hospital Central South University
11,mis_4a7ff343635312b3973ff97e121d1f6f,dim_ASC-OR-0000000078724-1.0-1724880256,1.0,chinese_clinical_trials_silver.trials,Xiangya Hospital Central South University


In [20]:
# Save final outputs
output_path = config.paths.FINAL_MATCHES
final_output.to_csv(output_path, index=False)
print(f"\nFinal matches saved to: {output_path}")

# Also save as parquet for faster loading
parquet_path = output_path.replace('.csv', '.parquet')
final_output.to_parquet(parquet_path, index=False)
print(f"Parquet version: {parquet_path}")



Final matches saved to: /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/final_matches.csv
Parquet version: /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/final_matches.parquet


In [5]:
from opensearchpy import OpenSearch

os_client = OpenSearch(
    hosts=[{'host': 'vpc-sci2-opensearch-qa-iys63fu7pnaarpatjgtwuxeyuq.us-east-1.es.amazonaws.com', 'port': 443}],
    http_auth=('allsci_admin', 'wyD2P052Jcdejntk!'),
    use_ssl=True,
    verify_certs=False,
    timeout=120
)

def get_org_ids_from_hierarchies():
    """Get all unique organization IDs from organization_hierarchies.v1"""
    ids = set()
    after_key = None
    
    while True:
        agg_body = {
            "composite": {
                "size": 10000,
                "sources": [
                    {"org_id": {"terms": {"field": "organizations.id"}}}
                ]
            }
        }
        if after_key:
            agg_body["composite"]["after"] = after_key
        
        response = os_client.search(
            index="organization_hierarchies.v1",
            body={
                "size": 0,
                "aggs": {"ids": agg_body}
            }
        )
        
        buckets = response['aggregations']['ids']['buckets']
        if not buckets:
            break
            
        for bucket in buckets:
            ids.add(bucket['key']['org_id'])
        
        after_key = response['aggregations']['ids'].get('after_key')
        if not after_key:
            break
        
        print(f"  hierarchies orgs: fetched {len(ids)} IDs so far...")
    
    return ids

# Get org IDs from both indices
org_ids = get_all_allsci_ids("organizations.v1")  # allsci_id from orgs
hierarchy_org_ids = get_org_ids_from_hierarchies()  # organizations.id from hierarchies

# Check coverage
in_org_not_hierarchy = org_ids - hierarchy_org_ids
in_hierarchy_not_org = hierarchy_org_ids - org_ids
overlap = org_ids & hierarchy_org_ids

print(f"organizations.v1 unique IDs: {len(org_ids)}")
print(f"Org IDs in hierarchies: {len(hierarchy_org_ids)}")
print(f"Overlap: {len(overlap)}")
print(f"In orgs but NOT in hierarchies: {len(in_org_not_hierarchy)}")
print(f"In hierarchies but NOT in orgs: {len(in_hierarchy_not_org)}")

/Users/robertlalani/miniconda3/envs/splink_entity_linking/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'vpc-sci2-opensearch-qa-iys63fu7pnaarpatjgtwuxeyuq.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  organizations.v1: fetched 10000 IDs so far...
  organizations.v1: fetched 20000 IDs so far...


/Users/robertlalani/miniconda3/envs/splink_entity_linking/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'vpc-sci2-opensearch-qa-iys63fu7pnaarpatjgtwuxeyuq.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/robertlalani/miniconda3/envs/splink_entity_linking/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'vpc-sci2-opensearch-qa-iys63fu7pnaarpatjgtwuxeyuq.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  organizations.v1: fetched 30000 IDs so far...


/Users/robertlalani/miniconda3/envs/splink_entity_linking/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'vpc-sci2-opensearch-qa-iys63fu7pnaarpatjgtwuxeyuq.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

/Users/robertlalani/miniconda3/envs/splink_entity_linking/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'vpc-sci2-opensearch-qa-iys63fu7pnaarpatjgtwuxeyuq.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  hierarchies orgs: fetched 20000 IDs so far...


/Users/robertlalani/miniconda3/envs/splink_entity_linking/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'vpc-sci2-opensearch-qa-iys63fu7pnaarpatjgtwuxeyuq.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  hierarchies orgs: fetched 23719 IDs so far...
organizations.v1 unique IDs: 241166
Org IDs in hierarchies: 23719
Overlap: 0
In orgs but NOT in hierarchies: 241166
In hierarchies but NOT in orgs: 23719


/Users/robertlalani/miniconda3/envs/splink_entity_linking/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'vpc-sci2-opensearch-qa-iys63fu7pnaarpatjgtwuxeyuq.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [7]:
def extract_core_id(full_id):
    """Extract core org ID: ASC-OR-XXXXXXXXXX from full ID"""
    # ASC-OR-0000000067013-1.0-1754933374 -> ASC-OR-0000000067013
    parts = full_id.split('-')
    if len(parts) >= 3:
        return '-'.join(parts[:3])  # ASC-OR-{number}
    return full_id

# Normalize both sets
org_core_ids = {extract_core_id(id) for id in org_ids}
hierarchy_core_ids = {extract_core_id(id) for id in hierarchy_org_ids}

# Check coverage with normalized IDs
in_org_not_hierarchy = org_core_ids - hierarchy_core_ids
in_hierarchy_not_org = hierarchy_core_ids - org_core_ids
overlap = org_core_ids & hierarchy_core_ids

print(f"organizations.v1 unique core IDs: {len(org_core_ids)}")
print(f"Org core IDs in hierarchies: {len(hierarchy_core_ids)}")
print(f"Overlap: {len(overlap)}")
print(f"In orgs but NOT in hierarchies: {len(in_org_not_hierarchy)}")
print(f"In hierarchies but NOT in orgs: {len(in_hierarchy_not_org)}")

# Sample of missing if any
if in_hierarchy_not_org:
    print(f"\nSample in hierarchies but not orgs: {list(in_hierarchy_not_org)[:5]}")

organizations.v1 unique core IDs: 124002
Org core IDs in hierarchies: 23719
Overlap: 23023
In orgs but NOT in hierarchies: 100979
In hierarchies but NOT in orgs: 696

Sample in hierarchies but not orgs: ['ASC-OR-0000000129519', 'ASC-OR-0000000129181', 'ASC-OR-0000000129445', 'ASC-OR-0000000128987', 'ASC-OR-0000000129116']


In [21]:
# Analysis summary
print("\n" + "=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)

print(f"""
RESULTS SUMMARY:
  - Total predictions analyzed: {len(predictions_df):,}
  - High confidence (>0.95): {len(very_high):,}
  - Medium confidence (0.7-0.85): {len(medium):,}
  - Low confidence (0.5-0.7): {len(low):,}

QUALITY METRICS:
  - Records with multiple matches: {len(multiple_matches):,}

OUTPUT FILES:
  - Final matches (CSV): {config.paths.FINAL_MATCHES}
  - Final matches (Parquet): {parquet_path}

RECOMMENDATIONS:
  - High confidence matches can be used directly
  - Medium confidence matches should be reviewed
  - Low confidence matches are likely false positives
""")

print("Analysis complete!")



ANALYSIS COMPLETE

RESULTS SUMMARY:
  - Total predictions analyzed: 493
  - High confidence (>0.95): 480
  - Medium confidence (0.7-0.85): 0
  - Low confidence (0.5-0.7): 1

QUALITY METRICS:
  - Records with multiple matches: 51

OUTPUT FILES:
  - Final matches (CSV): /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/final_matches.csv
  - Final matches (Parquet): /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/final_matches.parquet

RECOMMENDATIONS:
  - High confidence matches can be used directly
  - Medium confidence matches should be reviewed
  - Low confidence matches are likely false positives

Analysis complete!
